# Multiscale Bioparticle Transport & Membrane Fouling Framework
### End-to-End Demo: From Martini 3 CG-MD Closures to Continuum TFF Filtration Predictions

This interactive notebook demonstrates:
1. Ingestion of physical properties extracted from **Martini 3 Coarse-Grained MD** (`data/sample_md_params.json`).
2. Evaluation of constitutive closures (Carman-Kozeny cake resistance, Virial osmotic pressure, Krieger-Dougherty viscosity).
3. Simulation of Tangential Flow Filtration (TFF) and membrane fouling dynamics.
4. Publication-grade visualization of permeate flux decline $J(t)$, concentration polarization $C_w(t)$, and cake layer growth $R_c(t)$.

In [ ]:
import sys
from pathlib import Path
import matplotlib.pyplot as plt
import numpy as np

# Ensure package is in path
sys.path.insert(0, str(Path.cwd().parent / "02_continuum_transport" / "python"))
from biotransport.bridge import MdBridgeModel, ProcessSimulator, run_continuum_simulation
from biotransport.visualize import plot_multiscale_results

print("Successfully imported biotransport framework!")

## 1. Load Bridged Parameters from CG-MD

In [ ]:
params_path = Path.cwd().parent / "data" / "sample_md_params.json"
md_model = MdBridgeModel.load_json(params_path)

print("=== Molecular Dynamics Microscale Parameters ===")
print(f"Source Model:         {md_model.metadata.get('model', 'N/A')}")
print(f"Temperature:          {md_model.thermodynamics.temperature_K} K")
print(f"Hydrodynamic Radius:  {md_model.microscale_properties.hydrodynamic_radius_Rh_nm} nm")
print(f"Particle Density:     {md_model.microscale_properties.particle_density_kg_m3} kg/m^3")
print(f"Diffusivity (D0):     {md_model.microscale_properties.diffusion_coefficient_D0_m2_s:.2e} m^2/s")
print(f"Virial Coeff (B2):    {md_model.microscale_properties.osmotic_virial_B2_m3_mol:.2e} m^3/mol")

## 2. Run Continuum Filtration Simulation Across Transmembrane Pressures (TMP)

In [ ]:
sim = ProcessSimulator(md_model)

# Compare multiple TMP operating conditions (1.0 bar, 1.5 bar, 2.0 bar, 2.5 bar)
tmps = [100_000, 150_000, 200_000, 250_000]
labels = ["1.0 bar", "1.5 bar", "2.0 bar", "2.5 bar"]
results_dict = {}

for tmp, label in zip(tmps, labels):
    results_dict[label] = sim.simulate_filtration(tmp_pa=tmp, bulk_conc_g_l=10.0, total_time_s=3600.0)

fig, ax = plt.subplots(figsize=(8, 5), dpi=150)
for label, res in results_dict.items():
    t_min = np.array(res["time_s"]) / 60.0
    ax.plot(t_min, res["flux_lmh"], lw=2.2, label=f"TMP = {label}")

ax.set_xlabel("Filtration Time (minutes)", fontsize=12)
ax.set_ylabel("Permeate Flux (LMH)", fontsize=12)
ax.set_title("Flux Decline Dynamics vs Transmembrane Pressure", fontsize=13, fontweight="bold")
ax.grid(True, linestyle="--", alpha=0.6)
ax.legend(frameon=True)
plt.show()

## 3. Generate Complete 3-Panel Multiscale Overview Figure

In [ ]:
std_results = sim.simulate_filtration(tmp_pa=150_000.0, bulk_conc_g_l=10.0, total_time_s=3600.0)
plot_multiscale_results(std_results, output_png="../data/filtration_summary_figure.png")
print("Figure generated and saved to data/filtration_summary_figure.png")